In [1]:
import bw2data, bw2io, bw2calc
from bw_timex import TimexLCA
from bw_temporalis import TemporalDistribution, easy_timedelta_distribution
import numpy as np
from datetime import datetime
import os
import re
import pandas as pd
import numpy as np
import pickle

In [2]:
import sys
sys.path.append('../utils/') 
from elec_builder import *

In [3]:
# activate the bw project
bw2data.projects.set_current("ei311")
#for db in bw2data.databases:
#    print(db, len(bw2data.Database(db)))

#### for CN reservoir, using RoW as proxy
#### for US reservoir, using CA-QC as proxy 

In [4]:
#hydro_cn = ['CN-SC']

# for CN reservoir, using RoW as proxy
hydro_us = ['US-WECC']
hydro_qc = ["CA-QC"]
hydro_row = ['RoW']

xx = build_dynamic_electricity_all(
    locations = hydro_row , 
    pathways = [ "SSP1-VLLO" , "SSP2-M", "SSP5-H" ],
    years = [2030, 2040, 2050], 
    elec_act = 'electricity production, hydro, reservoir, non-alpine region', #'electricity production, hydro, run-of-river',
    ref_name = 'market for electricity, hydro, high voltage',
    fg_db_name="elec_hydro_reservoir_foreground",
    flush_fg_db = True
)

# not tested: assign_td_from_results_dict(results_dict = xx, elec_td_year=10)

pathways should match the premise database names, e.g., SSP1-VLLO, SSP2-M, SSP5-H 
 also IAM locations for premise database can be found through calling: 
 from premise import geomap -> premise.geomap.Geomap('image').iam_regions 
Created fresh foreground DB: 'elec_hydro_reservoir_foreground'


In [5]:
yy =  build_dynamic_electricity_all(
    locations = hydro_qc,
    pathways = [ "SSP1-VLLO" , "SSP2-M", "SSP5-H" ],
    years = [2030, 2040, 2050], 
    elec_act = 'electricity production, hydro, reservoir, non-alpine region',
    ref_name = 'market for electricity, hydro, high voltage',
    fg_db_name="elec_hydro_reservoir_foreground",
    flush_fg_db = False
)

pathways should match the premise database names, e.g., SSP1-VLLO, SSP2-M, SSP5-H 
 also IAM locations for premise database can be found through calling: 
 from premise import geomap -> premise.geomap.Geomap('image').iam_regions 
Using foreground DB: elec_hydro_reservoir_foreground, with 9 activities currently.


In [6]:
hydro_db = bw2data.Database("elec_hydro_reservoir_foreground")
len(hydro_db)

18

In [7]:
list(hydro_db)

['market for electricity, hydro, high voltage, CA-QC, SSP5-H, 2030' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP5-H, 2040' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP2-M, 2050' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, RoW, SSP1-VLLO, 2040' (kWh, RoW, None),
 'market for electricity, hydro, high voltage, RoW, SSP1-VLLO, 2030' (kWh, RoW, None),
 'market for electricity, hydro, high voltage, RoW, SSP1-VLLO, 2050' (kWh, RoW, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP1-VLLO, 2030' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP2-M, 2030' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, RoW, SSP5-H, 2030' (kWh, RoW, None),
 'market for electricity, hydro, high voltage, RoW, SSP2-M, 2030' (kWh, RoW, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP5-H, 2050' (kWh, CA-QC, None),
 'market for electricity, hydro, high vo

In [8]:
act_list = list(hydro_db)[2:4]
for act in act_list:
    tech_excs = list(act.technosphere())
    print(len(tech_excs))
    exc = tech_excs[0]
    print(exc)

1
Exchange: 1 kilowatt hour 'electricity production, hydro, reservoir, non-alpine region' (kilowatt hour, RoW, None) to 'market for electricity, hydro, high voltage, RoW, SSP1-VLLO, 2030' (kWh, RoW, None)>
1
Exchange: 1 kilowatt hour 'electricity production, hydro, reservoir, non-alpine region' (kilowatt hour, RoW, None) to 'market for electricity, hydro, high voltage, RoW, SSP5-H, 2040' (kWh, RoW, None)>


In [9]:
rows = []   # collect results for all acts
act_list = list(hydro_db)   
for act in act_list: 
    print(act)

    name = act.get("name")
    name_parts = [p.strip() for p in name.split(",")]
    # run static LCI + premise_GWP vs. pGWP100 first: 
    pgwp_fixedco2 = find_dpGWP100_method(name_parts[-2], int(name_parts[-1]), method_suffix = "pGWP100 - fixed-AGWPCO2")
    pgwp_dpco2 = find_dpGWP100_method(name_parts[-2], int(name_parts[-1]), method_suffix = "pGWP100 - dp-AGWPCO2")
    gwp =  ('ecoinvent-3.11', 'IPCC 2021 (incl. biogenic CO2)', 'climate change: total (incl. biogenic CO2, incl. SLCFs)', 'global warming potential (GWP100)')   

    lca_gwp = bw2calc.lca.LCA({act: 1}, method=gwp)
    lca_gwp.lci(); lca_gwp.lcia()
    score_gwp = float(lca_gwp.score)

    lca_fixed = bw2calc.lca.LCA({act: 1}, method=pgwp_fixedco2)
    lca_fixed.lci(); lca_fixed.lcia()
    score_fixedco2 = float(lca_fixed.score)

    lca_dp = bw2calc.lca.LCA({act: 1}, method=pgwp_dpco2)
    lca_dp.lci(); lca_dp.lcia()
    score_dpco2 = float(lca_dp.score)

    print(score_gwp, score_fixedco2, score_dpco2) 

    # ---- store results for this activity ----
    rows.append({
        "Activity": name,                      # index value later
        "gwp100": score_gwp,
        "pGWP100_fixedCO2": score_fixedco2,
        "pGWP100_dpCO2": score_dpco2,
    })


'market for electricity, hydro, high voltage, RoW, SSP2-M, 2030' (kWh, RoW, None)
0.0506240716426115 0.05584781277644521 0.04991070284626162
'market for electricity, hydro, high voltage, RoW, SSP2-M, 2040' (kWh, RoW, None)
0.05016714038066257 0.05256302093778377 0.04990881578455679
'market for electricity, hydro, high voltage, CA-QC, SSP5-H, 2030' (kWh, CA-QC, None)
0.0158012883553034 0.017236325178465225 0.015666238870664376
'market for electricity, hydro, high voltage, RoW, SSP5-H, 2040' (kWh, RoW, None)
0.050594952887580834 0.04985878762114201 0.04968123964054657
'market for electricity, hydro, high voltage, RoW, SSP2-M, 2050' (kWh, RoW, None)
0.04984364079505242 0.04970101534488699 0.04993021028670728
'market for electricity, hydro, high voltage, RoW, SSP1-VLLO, 2050' (kWh, RoW, None)
0.04743668155690239 0.05397877348149437 0.04740237046169956
'market for electricity, hydro, high voltage, CA-QC, SSP2-M, 2040' (kWh, CA-QC, None)
0.015561841109323798 0.016342932251011334 0.0155218693

In [10]:
df_scores = pd.DataFrame(rows)
df_scores = df_scores.set_index("Activity")

df2 = df_scores.sort_values(by=['Activity'])
df2

,gwp100,pGWP100_fixedCO2,pGWP100_dpCO2
Activity,,,
"market for electricity, hydro, high voltage, CA-QC, SSP1-VLLO, 2030",0.015721,0.017909,0.015650
"market for electricity, hydro, high voltage, CA-QC, SSP1-VLLO, 2040",0.015028,0.016988,0.015003
"market for electricity, hydro, high voltage, CA-QC, SSP1-VLLO, 2050",0.013658,0.015535,0.013646
"market for electricity, hydro, high voltage, CA-QC, SSP2-M, 2030",0.015746,0.017505,0.015649
"market for electricity, hydro, high voltage, CA-QC, SSP2-M, 2040",0.015562,0.016343,0.015522
"market for electricity, hydro, high voltage, CA-QC, SSP2-M, 2050",0.015461,0.015388,0.015462
"market for electricity, hydro, high voltage, CA-QC, SSP5-H, 2030",0.015801,0.017236,0.015666
"market for electricity, hydro, high voltage, CA-QC, SSP5-H, 2040",0.015678,0.015613,0.015559
"market for electricity, hydro, high voltage, CA-QC, SSP5-H, 2050",0.015582,0.014036,0.015528


In [12]:
df2.index = df2.index.str.replace("RoW", "CN", regex=False)
df2.index = df2.index.str.replace("CA-QC", "US", regex=False)
df2

,gwp100,pGWP100_fixedCO2,pGWP100_dpCO2
Activity,,,
"market for electricity, hydro, high voltage, US, SSP1-VLLO, 2030",0.015721,0.017909,0.015650
"market for electricity, hydro, high voltage, US, SSP1-VLLO, 2040",0.015028,0.016988,0.015003
"market for electricity, hydro, high voltage, US, SSP1-VLLO, 2050",0.013658,0.015535,0.013646
"market for electricity, hydro, high voltage, US, SSP2-M, 2030",0.015746,0.017505,0.015649
"market for electricity, hydro, high voltage, US, SSP2-M, 2040",0.015562,0.016343,0.015522
"market for electricity, hydro, high voltage, US, SSP2-M, 2050",0.015461,0.015388,0.015462
"market for electricity, hydro, high voltage, US, SSP5-H, 2030",0.015801,0.017236,0.015666
"market for electricity, hydro, high voltage, US, SSP5-H, 2040",0.015678,0.015613,0.015559
"market for electricity, hydro, high voltage, US, SSP5-H, 2050",0.015582,0.014036,0.015528


In [13]:
df2.to_excel("dp-LCI_output/staticLCI_(p)GWP100/hydro_reservoir_CN-asRoW_US-asQC_staticLCI_threeGWP100.xlsx")

### 2. building dpLCI

In [4]:
hydro_db = bw2data.Database("elec_hydro_reservoir_foreground")
list(hydro_db)

['market for electricity, hydro, high voltage, RoW, SSP5-H, 2050' (kWh, RoW, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP2-M, 2050' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP1-VLLO, 2050' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, RoW, SSP1-VLLO, 2030' (kWh, RoW, None),
 'market for electricity, hydro, high voltage, RoW, SSP2-M, 2050' (kWh, RoW, None),
 'market for electricity, hydro, high voltage, RoW, SSP5-H, 2040' (kWh, RoW, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP5-H, 2040' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP5-H, 2030' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP2-M, 2040' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP1-VLLO, 2030' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP1-VLLO, 2040' (kWh, CA-QC, None),
 'market for electricity, hydro, hig

In [5]:
hydro_db_203050 = [
    act for act in hydro_db 
    if "2050" in str(act.get('name', '')) or  "2030" in str(act.get('name', ''))
]

hydro_db_203050

['market for electricity, hydro, high voltage, RoW, SSP1-VLLO, 2050' (kWh, RoW, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP1-VLLO, 2030' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP2-M, 2030' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP5-H, 2050' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP5-H, 2030' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP2-M, 2050' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, RoW, SSP5-H, 2030' (kWh, RoW, None),
 'market for electricity, hydro, high voltage, CA-QC, SSP1-VLLO, 2050' (kWh, CA-QC, None),
 'market for electricity, hydro, high voltage, RoW, SSP5-H, 2050' (kWh, RoW, None),
 'market for electricity, hydro, high voltage, RoW, SSP2-M, 2030' (kWh, RoW, None),
 'market for electricity, hydro, high voltage, RoW, SSP2-M, 2050' (kWh, RoW, None),
 'market for electricity, hydro, high volta

In [8]:
database_dates = {
    'ei_cutoff_3.11_image_SSP1-VLLO_2030 2025-11-24': datetime.strptime("2030", "%Y"),
    'ei_cutoff_3.11_image_SSP1-VLLO_2040 2025-11-22': datetime.strptime("2040", "%Y"),
    'ei_cutoff_3.11_image_SSP1-VLLO_2050 2025-11-22': datetime.strptime("2050", "%Y"),

    'ei_cutoff_3.11_image_SSP2-M_2030 2025-11-22': datetime.strptime("2030", "%Y"),
    'ei_cutoff_3.11_image_SSP2-M_2040 2025-11-22': datetime.strptime("2040", "%Y"),
    'ei_cutoff_3.11_image_SSP2-M_2050 2025-11-22': datetime.strptime("2050", "%Y"),

    'ei_cutoff_3.11_image_SSP5-H_2030 2025-11-22': datetime.strptime("2030", "%Y"),
    'ei_cutoff_3.11_image_SSP5-H_2040 2025-11-22': datetime.strptime("2040", "%Y"),
    'ei_cutoff_3.11_image_SSP5-H_2050 2025-11-22': datetime.strptime("2050", "%Y"),
    
    "elec_hydro_reservoir_foreground": "dynamic", # flag databases that should be temporally distributed with "dynamic"
}

In [9]:
dp_results = {}

for act in list(hydro_db_203050): 
    print(act)
 
    #### dynamic LCI:: 
    assign_td_from_foreground_db( 
        select_act = act,
        elec_td_year=10,
        resolution="Y",
        kind="uniform",
        fg_db_name="elec_hydro_reservoir_foreground",
        verbose = True
     )

    tlca = run_dp_timex_lca(foreground_act = act ,   
                    pathway = None,
                    year = None,
                    method = None,
                    database_dates = database_dates, #None not working, has to incl. all 9 background DB ... 
                    temporal_grouping="year", 
                    method_prefix = "Climate Change prospective GWP100",
                    method_suffix = "pGWP100 - fixed-AGWPCO2", 
                    fg_db_name = 'elec_hydro_reservoir_foreground'
     )

    tlca.lci()
    tlca.dynamic_inventory.shape

    # dyn-foreground LCI + static background LCI + pGWP100
    lca_0 = tlca.base_score 
    print(lca_0)
    # dpLCI (dyn-foreground LCI + dyn background LCI) +  pGWP100
    tlca.static_lcia()
    lca_1 = tlca.static_score
    print(lca_1)

    act_name = act.get("name")
    dp_results[act_name] = {
        "dyn FG LCI static BG p-LCI, pGWP100-fixedCO2": float(lca_0),   # static BG LCI + dyn FG LCI
        "full dp-LCI, pGWP100-fixedCO2": float(lca_1),      # dyn BG LCI + dyn FG LCI
    }

    #### now save all dyLCI flows to pandas 
    df = tlca.dynamic_inventory_df

    ##### important to convert flow and act as str to excel 
    df["flow"] = df["flow"].astype(str)
    df["activity"] = df["activity"].astype(str)

    ### export df to dp-LCI_output folder, using the act name as the excel name 
    out_dir = "dp-LCI_output/hydro_dpLCI_v2_reservoir"
    os.makedirs(out_dir, exist_ok=True)    
    raw_name = act["name"]
    safe_name = re.sub(r"[^A-Za-z0-9_\-()]+", "_", raw_name)   # replace spaces/special chars
    
    excel_path = os.path.join(out_dir, f"{safe_name}.xlsx")
    
    df.to_excel(excel_path, index=False)    
    print(f"✔ Exported dynamic inventory DF for '{raw_name}' → {excel_path}")


import pickle

out_dir = "dp-LCI_output/hydro_dpLCI_v2_reservoir"
os.makedirs(out_dir, exist_ok=True)

pickle_path = os.path.join(out_dir, "hydro_v2_reservoir_dp_results_MY2030_2050.pkl")

with open(pickle_path, "wb") as f:
    pickle.dump(dp_results, f)

print(f"✔ Saved dp_results dictionary → {pickle_path}")

2025-12-11 19:04:22.528 | INFO     | bw_timex.timex_lca:__init__:114 - Initializing TimexLCA object...
2025-12-11 19:04:22.528 | INFO     | bw_timex.timex_lca:__init__:136 - Collecting node infos...


'market for electricity, hydro, high voltage, RoW, SSP1-VLLO, 2050' (kWh, RoW, None)
TD applied to 'market for electricity, hydro, high voltage, RoW, SSP1-VLLO, 2050' → Exchange: 1 kilowatt hour 'electricity production, hydro, reservoir, non-alpine region' (kilowatt hour, RoW, None) to 'market for electricity, hydro, high voltage, RoW, SSP1-VLLO, 2050' (kWh, RoW, None)>
for the activity 'market for electricity, hydro, high voltage, RoW, SSP1-VLLO, 2050' (kWh, RoW, None),it's under SSP-SSP1-VLLO, year-2050 
 we'll use LCIA ('Climate Change prospective GWP100', 'SSP119', 'MY2050', 'pGWP100 - fixed-AGWPCO2')  with background database.datetime = {'ei_cutoff_3.11_image_SSP1-VLLO_2030 2025-11-24': datetime.datetime(2030, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP1-VLLO_2040 2025-11-22': datetime.datetime(2040, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP1-VLLO_2050 2025-11-22': datetime.datetime(2050, 1, 1, 0, 0), 'ei_cutoff_3.11_image_SSP2-M_2030 2025-11-22': datetime.datetime(2030, 1, 1, 0, 0), 'ei_

2025-12-11 19:08:04.689 | INFO     | bw_timex.timex_lca:build_timeline:216 - No edge filter function provided. Skipping all edges in background databases.
2025-12-11 19:09:50.328 | INFO     | bw_timex.timex_lca:build_timeline:232 - Calculating base LCA...
2025-12-11 19:10:26.260 | INFO     | bw_timex.timex_lca:build_timeline:242 - Creating activity time mapping...
2025-12-11 19:10:29.719 | INFO     | bw_timex.timeline_builder:__init__:99 - Traversing supply chain graph...


Starting graph traversal


2025-12-11 19:10:31.734 | INFO     | bw_timex.timeline_builder:build_timeline:142 - Building timeline...


...

Starting graph traversal


2025-12-11 21:27:38.404 | INFO     | bw_timex.timeline_builder:build_timeline:142 - Building timeline...


Calculation count: 1


2025-12-11 21:27:38.587 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2026-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-11 21:27:38.591 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2026-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-11 21:27:38.595 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2027-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-11 21:27:38.597 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:522 - Reference date 2028-01-01 00:00:00 is lower than all provided dates. Data will be taken from the closest higher year.
2025-12-11 21:27:38.598 | INFO     | bw_time

0.05726559498803554
0.057505282300774264
✔ Exported dynamic inventory DF for 'market for electricity, hydro, high voltage, RoW, SSP1-VLLO, 2030' → dp-LCI_output/hydro_dpLCI_v2_reservoir/market_for_electricity_hydro_high_voltage_RoW_SSP1-VLLO_2030.xlsx
✔ Saved dp_results dictionary → dp-LCI_output/hydro_dpLCI_v2_reservoir/hydro_v2_reservoir_dp_results_MY2030_2050.pkl
